# Output evaluator
Last update: _2026-11-17_

- Output evaluator
    - question: -
    - retrieved chunks: _"Chunking"_, _"Retrieval"_ and _"Reranking"_  
    - answer: _"Generation"_

Building the knowledge base
```
python build_kb_csv.py --config configs/config.yaml --csv C:\Users\Zol\Documents\minimal-RAG-test-set\data\export\laudon_export.csv --source-col source --chunks-col chunks --chapter-col chapter
```

Batch provessing the test set
```
py main_csv.py
```

__Todo__
- [x] Test set generator
- [x] Agent
    - [x] Add Laudon to FAISS
    - [x] Run question against the agent
- [ ] Output evaluator
    - [ ] Retrieval
        - [ ] Context Precision
        - [ ] [Content Recall](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_recall/)
        - [ ] Context Entities Recall
        - [ ] Context Relevance
        - [ ] [Mean Reciprocal Rank](https://amenra.github.io/ranx/metrics/#mean-reciprocal-rank)
        - [ ] Normalized Discounted Cumulative Gain
        - [ ] Mean Average Precision
    - [ ] Generation
        - [ ] Response Groundedness
        - [ ] Faithfulness
        - [ ] Answer Accuracy
        - [ ] Factual Correctness
        - [ ] BLEU Score
        - [ ] CHRF Score
        - [ ] Rouge Score
    - [ ] End-to-end
        - [ ] Noise sensitivity
        - [ ] Response relevancy
        - [ ] Semantic Similarity
    - [x] Check output
    - [x] Modify output format
    - [x] Use LLM config
    - [x] Document workflow
    - [x] Push to Git
    - [ ] Fix metric generation
    - [ ] Fork Flora's Git
- [ ] Result visualizer

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../data/agent_output/laudon_questions_with_answers.csv")

In [ ]:
df[:5]

## Loading LLM

In [ ]:
import os
import time
import yaml
import litellm
from pathlib import Path
from pydantic import BaseModel

In [ ]:
def load_llm_config(
    config_path: Optional[str],
    entry: str,
) -> str:
    """
    Load model name (and optionally set API key) from a YAML config file.

    Expected structure:
        claude_5_5:
          provider: claude-haiku-5-5   # or any litellm model string
          api_key: sk-...
    """
    if config_path is None:
        # Fall back to environment
        model = os.environ.get("LLM_MODEL")
        if not model:
            raise ValueError(
                "No --llm-config provided and LLM_MODEL environment variable is not set."
            )
        return model

    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"LLM config file not found: {config_path}")

    with open(path, "r", encoding="utf-8") as fh:
        config = yaml.safe_load(fh)

    if entry not in config:
        raise KeyError(
            f"Entry '{entry}' not found in {config_path}. "
            f"Available keys: {list(config.keys())}"
        )

    entry_cfg = config[entry]
    model = entry_cfg.get("provider") or entry_cfg.get("model")
    if not model:
        raise ValueError(f"No 'provider' or 'model' key under '{entry}' in config.")

    # Set common API keys if present
    if "api_key" in entry_cfg:
        # Heuristic: Anthropic keys start with sk-ant- or are long; OpenAI with sk-
        key = entry_cfg["api_key"]
        if "anthropic" in model.lower() or "claude" in model.lower():
            os.environ["ANTHROPIC_API_KEY"] = key
        else:
            os.environ.setdefault("OPENAI_API_KEY", key)

    return model

In [ ]:
llm_model = load_llm_config('../configs/llm.yaml', 'claude_5_5')

In [ ]:
import os
from dataclasses import dataclass

import litellm
from ragas.llms import llm_factory
from ragas import EvaluationDataset, evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference,
)

In [ ]:


@dataclass
class LLMConfig:
    model: str
    temperature: float = 0.0
    max_tokens: int = 4096


def create_evaluation_llm(config: LLMConfig):
    """Create a RAGAS evaluation LLM using LiteLLM."""

    return llm_factory(
        model=config.model,
        provider="litellm",
        client=litellm.completion,
        adapter="litellm",
        temperature=config.temperature,
        max_tokens=config.max_tokens,
    )


def main():
    # Configure the evaluator model
    config = LLMConfig(
        model=os.getenv(
            "EVAL_MODEL",
            "openai/gpt-4o-mini",
        )
    )

    evaluator_llm = create_evaluation_llm(config)

    # Example RAG evaluation dataset
    samples = [
        {
            "user_input": "What is RAG?",
            "retrieved_contexts": [
                "Retrieval-Augmented Generation (RAG) "
                "combines information retrieval with "
                "generative language models."
            ],
            "response": (
                "RAG combines document retrieval "
                "with a generative language model."
            ),
        },
        {
            "user_input": "What is the capital of France?",
            "retrieved_contexts": [
                "Paris is the capital and largest city of France."
            ],
            "response": "The capital of France is Paris.",
        },
    ]

    dataset = EvaluationDataset.from_list(samples)

    # Configure metrics
    metrics = [
        Faithfulness(llm=evaluator_llm),
        ResponseRelevancy(llm=evaluator_llm),
        LLMContextPrecisionWithoutReference(llm=evaluator_llm),
    ]

    # Run evaluation
    results = evaluate(
        dataset=dataset,
        metrics=metrics,
    )

    print(results)
    print(results.to_pandas())

In [ ]:
call_llm('What is the capital of Paris?', llm_model)

## Evaluation

In [ ]:
from tqdm import tqdm

In [ ]:
for i in tqdm(range(df.shape[0])):
    

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision

In [ ]:
client = AsyncOpenAI()
llm = llm_factory("gpt-4o-mini", client=client)

scorer = ContextPrecision(llm=llm)

In [ ]:


# Setup LLM
client = AsyncOpenAI()
llm = llm_factory("gpt-4o-mini", client=client)

# Create metric


# Evaluate
result = await scorer.ascore(
    user_input="Where is the Eiffel Tower located?",
    reference="The Eiffel Tower is located in Paris.",
    retrieved_contexts=[
        "The Eiffel Tower is located in Paris.",
        "The Brandenburg Gate is located in Berlin.",
    ],
)
print(f"Context Precision Score: {result.value}")